In [1]:
import os
os.environ["TF_FORCE_GPU_ALLOW_GROWTH"] = "true"

from tscglue import data_loader
data_loader.DATA_DIR = "/home/petelin/TSCbench/data"

/home/petelin/TSCbench/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import numpy as np
from sklearn.metrics import accuracy_score
#from tscglue.models import LokyStackerV10FM
#from tscglue import data_loader
import polars as pl
from sklearn.metrics import log_loss

In [3]:
from tscbench.utils import load_ucr_fold


dataset = "Worms"
# dataset = 'Car'
# dataset = 'HandOutlines'
#dataset = 'Trace'
#dataset = 'SwedishLeaf'
#dataset = 'Meat'
# dataset='ACSF1'
# dataset='ElectricDevices'
X_train, y_train, X_test, y_test = load_ucr_fold(dataset, 19)
print(X_train.shape, y_train.shape, X_test.shape, y_test.shape)

(181, 1, 900) (181,) (77, 1, 900) (77,)


In [4]:
seed = 2683

In [5]:
from tscglue.models import LokyStackerV10RSTSFRandom, TSCGlueClassifier

m = TSCGlueClassifier(random_state=seed, n_jobs=16, verbose=10)

In [6]:
m.fit(X_train, y_train)

[0.00s] Starting fit, run_dir=tscglue_runs/08dbd768365346fe, n_jobs=16
[0.00s] CPUs set/available/used/ 16/48/16
[0.02s] GPUs set/available[torch]/available[smi]/used/ 0/0/2/0
[0.02s] Saved X and y to disk in 0.00s (dtype=float64)
[1.94s] Fit+transformed multirocket_s_1675711793 features (181, 49728) (68.67 MB) dtype=float64 in 1.8134s
[subprocess] fit_transform hydra_s_2109509380: 1.4659s
[7.23s] Fit+transformed hydra_s_2109509380 features (181, 7168) (9.90 MB) dtype=float64 in 5.2910s
[subprocess] fit_transform quant: 0.2969s
[10.79s] Fit+transformed quant features (181, 7987) (11.03 MB) dtype=float64 in 3.5602s
[15.27s] Fit+transformed rdst_s_1973074453 features (181, 30000) (41.43 MB) dtype=float64 in 4.4725s
[19.75s] Fit+transformed rstsf-random_s_447740098 features (181, 13622) (18.81 MB) dtype=float64 in 4.4816s
[subprocess] fit_transform mantis: 2.0269s
[25.03s] Fit+transformed mantis features (181, 512) (0.71 MB) dtype=float64 in 5.2849s
[subprocess] fit_transform chronos2: 14

,random_state,2683
,k_folds,10
,n_jobs,16
,verbose,10
,n_repetitions,1
,n_gpus,0


In [7]:
pl.DataFrame(m.summary(return_transforms=True)).head(8)

model,level,oof_accuracy,train_time
str,i64,f64,list[f64]
"""multirocket_s_1675711793""",null,null,[1.813399]
"""hydra_s_2109509380""",null,null,[5.29096]
"""quant""",null,null,[3.560234]
"""rdst_s_1973074453""",null,null,[4.472528]
"""rstsf-random_s_447740098""",null,null,[4.481613]
"""mantis""",null,null,[5.284875]
"""chronos2""",null,null,[18.005729]
"""multirockethydra-bestk-p-ridge…",0,0.751381,"[0.802684, 0.811441, … 0.919782]"


In [8]:
m.classes_

array(['1', '2', '3', '4', '5'], dtype='<U1')

In [9]:
y_pred = m.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f"Accuracy on {dataset}: {acc}")

[0.00s] Starting prediction
[0.73s] Computed multirocket_s_1675711793 features (77, 49728) (29.21 MB) dtype=float64 in 0.6797s
[subprocess] transform hydra_s_2109509380: 0.5034s
[5.42s] Computed hydra_s_2109509380 features (77, 7168) (4.21 MB) dtype=float64 in 4.6866s
[subprocess] transform quant: 0.2162s
[8.83s] Computed quant features (77, 7987) (4.69 MB) dtype=float64 in 3.4158s
[10.19s] Computed rdst_s_1973074453 features (77, 30000) (17.62 MB) dtype=float64 in 1.3610s
[11.77s] Computed rstsf-random_s_447740098 features (77, 13622) (8.00 MB) dtype=float64 in 1.5760s
[subprocess] transform mantis: 1.7528s
[16.79s] Computed mantis features (77, 512) (0.30 MB) dtype=float64 in 5.0191s
[subprocess] transform chronos2: 7.9093s
[27.98s] Computed chronos2 features (77, 1536) (0.90 MB) dtype=float64 in 11.1857s
[27.98s] Computed and saved features for prediction
[27.98s] Starting prediction with 16 workers for 50 first-level models
[28.02s] Predicted quant-etc_s_1672401463 in 0.0239s
[28.0

In [10]:
proba = m.predict_proba(X_test)
classes = list(m.classes_)

print(f"Log-loss: {log_loss(y_test, proba, labels=classes):.4f}")
# print(f"AUC (OvR): {roc_auc_score(y_test, proba, multi_class='ovr', labels=classes):.4f}")

[0.00s] Starting prediction
[0.66s] Computed multirocket_s_1675711793 features (77, 49728) (29.21 MB) dtype=float64 in 0.6252s
[subprocess] transform hydra_s_2109509380: 0.5110s
[5.22s] Computed hydra_s_2109509380 features (77, 7168) (4.21 MB) dtype=float64 in 4.5619s
[subprocess] transform quant: 0.1996s
[8.66s] Computed quant features (77, 7987) (4.69 MB) dtype=float64 in 3.4351s
[9.97s] Computed rdst_s_1973074453 features (77, 30000) (17.62 MB) dtype=float64 in 1.3147s
[11.50s] Computed rstsf-random_s_447740098 features (77, 13622) (8.00 MB) dtype=float64 in 1.5271s
[subprocess] transform mantis: 1.7591s
[16.77s] Computed mantis features (77, 512) (0.30 MB) dtype=float64 in 5.2703s
[subprocess] transform chronos2: 8.2785s
[28.34s] Computed chronos2 features (77, 1536) (0.90 MB) dtype=float64 in 11.5636s
[28.34s] Computed and saved features for prediction
[28.34s] Starting prediction with 16 workers for 50 first-level models
[28.38s] Predicted quant-etc_s_1672401463 in 0.0228s
[28.38

In [11]:
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.linear_model import RidgeClassifierCV
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.preprocessing import LabelEncoder
import matplotlib.pyplot as plt

# Build probability matrix from level-0 neki predictions
df = pl.DataFrame(m.neki).filter(pl.col("level") == 0)
prob_matrix = df.pivot(on="class", index=["index", "model"], values="probability", aggregate_function='mean')
class_cols = [c for c in prob_matrix.columns if c not in ("index", "model")]
wide = prob_matrix.pivot(on="model", index="index", values=class_cols)
wide = wide.sort("index")
X_stack = wide.drop("index").to_numpy()

le = LabelEncoder()
y_enc = le.fit_transform(y_train)

stackers = {
    "RidgeCV": RidgeClassifierCV(alphas=np.logspace(-3, 3, 20)),
    "RF": RandomForestClassifier(n_estimators=200, n_jobs=-1),
    "HGBT": HistGradientBoostingClassifier(max_iter=200),
    "MLP": MLPClassifier(hidden_layer_sizes=(100,), max_iter=500),
}

N_REPEATS = 20
rskf = RepeatedStratifiedKFold(n_splits=10, n_repeats=N_REPEATS, random_state=42)
results = {name: [] for name in stackers}

for fold_i, (tr_idx, val_idx) in enumerate(rskf.split(X_stack, y_enc)):
    rep = fold_i // 10
    for name, clf in stackers.items():
        clf_copy = clf.__class__(**clf.get_params())
        clf_copy.fit(X_stack[tr_idx], y_enc[tr_idx])
        acc = accuracy_score(y_enc[val_idx], clf_copy.predict(X_stack[val_idx]))
        results[name].append({"rep": rep, "fold": fold_i % 10, "acc": acc})

# Build test probability matrix
test_preds = m.predict_proba_per_model(X_test)
train_cols = wide.drop("index").columns

test_records = []
for model_name, proba_arr in test_preds.items():
    if model_name in m.stacking_models:
        continue
    classes_ = list(m.classes_)
    for i in range(proba_arr.shape[0]):
        for j, cls in enumerate(classes_):
            test_records.append({"index": i, "model": model_name, "level": 0, "class": str(cls), "probability": proba_arr[i, j]})

df_test = pl.DataFrame(test_records)
prob_matrix_test = df_test.pivot(on="class", index=["index", "model"], values="probability")
class_cols_test = [c for c in prob_matrix_test.columns if c not in ("index", "model")]
wide_test = prob_matrix_test.pivot(on="model", index="index", values=class_cols_test)
wide_test = wide_test.sort("index")
X_stack_test = wide_test.select(train_cols).to_numpy()
y_test_enc = le.transform(y_test)

# Test set accuracy
test_accs = {}
for name, clf in stackers.items():
    clf_final = clf.__class__(**clf.get_params())
    clf_final.fit(X_stack, y_enc)
    test_accs[name] = accuracy_score(y_test_enc, clf_final.predict(X_stack_test))

# Plot
fig, ax = plt.subplots(figsize=(10, 5))
names = list(stackers.keys())
x_positions = np.arange(len(names))

for i, name in enumerate(names):
    rdf = pl.DataFrame(results[name])
    per_rep = rdf.group_by("rep").agg(pl.col("acc").mean()).sort("rep")["acc"].to_numpy()
    # scatter all per-rep OOF accuracies
    jitter = np.random.default_rng(0).uniform(-0.15, 0.15, size=len(per_rep))
    ax.scatter(np.full_like(per_rep, i) + jitter, per_rep, alpha=0.5, s=40, zorder=3, label=f"{name} OOF reps" if i == 0 else None, color=f"C{i}")
    # mean OOF line
    ax.hlines(per_rep.mean(), i - 0.25, i + 0.25, colors=f"C{i}", linewidths=2, zorder=4)
    # test accuracy marker
    ax.scatter(i, test_accs[name], marker="*", s=200, color=f"C{i}", edgecolors="black", linewidths=0.8, zorder=5)
    print(f"{name:8s}  OOF mean={per_rep.mean():.4f} std={per_rep.std():.4f}  test={test_accs[name]:.4f}")

ax.set_xticks(x_positions)
ax.set_xticklabels(names)
ax.set_ylabel("Accuracy")
ax.set_title(f"Stacker comparison: OOF ({N_REPEATS}x10-fold) vs Test")
# custom legend
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], marker="o", color="w", markerfacecolor="gray", markersize=8, label="OOF per-rep mean"),
    Line2D([0], [0], marker="*", color="w", markerfacecolor="gray", markeredgecolor="black", markersize=15, label="Test accuracy"),
    Line2D([0], [0], color="gray", linewidth=2, label="OOF overall mean"),
]
ax.legend(handles=legend_elements, loc="lower right")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()


AttributeError: 'TSCGlueClassifier' object has no attribute 'neki'